# Step 4 → Step 5: 재화 소모 → 결제 전환 분석

## 1. 분석 배경
1차 행동 루프 분석 결과,
재화 소모 → 최초 결제 구간의 전환율은 10.63%로 나타남.

- Step 4 재화 소모 사용자: 3,367명
- Step 5 결제 사용자: 358명
- 결제 전환율: 10.63%

본 분석에서는 재화 소모 사용자 중
결제로 전환된 사용자와 전환되지 않은 사용자의 차이를 분석함.

## 2. 핵심 질문
- 어떤 재화 소모 행동을 가진 사용자가 결제로 전환되는가?
- 결제자와 미결제자는 어떤 행동 차이를 보이는가?
- 재화 소모 사용자들을 행동 특성에 따라 세그먼트화할 수 있는가?
- 결제 미전환의 원인으로 설명 가능한 행동 패턴은 무엇인가?

## 3. 분석 대상
- 결제 전환 사용자: 358명
- 미결제 사용자: 3,009명
- 전체 분석 대상: 3,367명

## 4. 역할 분담

### 재화 소모 행동 및 세그먼트 분석 (선)
- 재화 소모 횟수
- 총/평균/최대 소모량
- 소모 금액 유형
- 반복 소모 패턴
- 결제자/미결제자 비교
- 사용자 세그먼트 분석

### 가설 기반 분석 (중현)
- 결제 미전환 원인 가설 설정
- 무료 포인트 확보 가능성
- 투표 참여와 예상 포인트 획득량
- 행동 소요 시간
- 가설별 데이터 검증

## 5. 이후 진행
EDA 및 가설 분석
→ 주요 결제 전환 요인 도출
→ 필요 시 모델링
→ 개선안 및 A/B 테스트 설계

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# 프로젝트 루트 경로
PROJECT_ROOT = Path.cwd().parents[3]
DATA_DIR = PROJECT_ROOT / "data" / "processed"

In [3]:
# 공통 데이터 로드
friend_df = pd.read_csv(
    DATA_DIR / "accounts_friendrequest_preprocessed_2.csv"
)

question_df = pd.read_csv(
    DATA_DIR / "accounts_userquestionrecord_preprocessed_2.csv"
)

point_df = pd.read_csv(
    DATA_DIR / "accounts_pointhistory_preprocessed_2.csv"
)

payment_df = pd.read_csv(
    DATA_DIR / "accounts_paymenthistory_preprocessed_2.csv"
)

In [4]:
print("friend:", friend_df.shape)
print("question:", question_df.shape)
print("point:", point_df.shape)
print("payment:", payment_df.shape)

friend: (7793140, 6)
question: (800793, 12)
point: (88762, 5)
payment: (95137, 5)


In [5]:
# 1. 친구 관계 형성 사용자
send_users = friend_df["send_user_id"]
receive_users = friend_df["receive_user_id"]

friend_users = set(
    pd.concat(
        [send_users, receive_users],
        ignore_index=True
    )
    .dropna()
    .unique()
)

In [6]:
# 2. 투표 수신 사용자
vote_received_users = set(
    question_df["chosen_user_id"]
    .dropna()
    .unique()
)

In [7]:
# 3. 친구 관계 형성 + 투표 수신 사용자
vote_stage_users = (
    friend_users
    & vote_received_users
)

In [8]:
# 4. 재화 소모 사용자
# pointhistory_preprocessed_2는 이미 delta_point < 0 기준으로 재전처리된 데이터
spend_users = set(
    point_df["user_id"]
    .dropna()
    .unique()
)

In [9]:
# 5. Step 4 사용자
# 친구 관계 형성 → 투표 수신 → 재화 소모
step4_users = (
    vote_stage_users
    & spend_users
)


In [10]:
print("친구 관계 형성 사용자:", len(friend_users))
print("투표 수신 단계 사용자:", len(vote_stage_users))
print("Step 4 재화 소모 사용자:", len(step4_users))

친구 관계 형성 사용자: 574163
투표 수신 단계 사용자: 10647
Step 4 재화 소모 사용자: 3367


In [11]:
# 결제 사용자
payment_users = set(
    payment_df["user_id"]
    .dropna()
    .unique()
)

In [12]:
# Step 4 사용자 중 결제한 사용자
step5_users = (
    step4_users
    & payment_users
)

In [13]:
# 결제하지 않은 사용자
non_payment_users = (
    step4_users
    - payment_users
)

In [14]:
print("Step 4 재화 소모 사용자:", len(step4_users))
print("Step 5 결제 사용자:", len(step5_users))
print("미결제 사용자:", len(non_payment_users))
print(
    f"결제 전환율: "
    f"{len(step5_users) / len(step4_users) * 100:.2f}%"
)

Step 4 재화 소모 사용자: 3367
Step 5 결제 사용자: 358
미결제 사용자: 3009
결제 전환율: 10.63%
